# Fasjämvikter med van der Waals gaslag
[Ideala gaslagen (Wikipedia)](https://sv.wikipedia.org/wiki/Ideala_gaslagen)

Allmänna gaslagen säger att trycket $p$ [N/$m^2$] varierar med koncentrationen, $c = n/V$ [mol/L], och temperaturen, $T$ [K]. $R$ är den allmänna gaskonstanten.

$p = 1000\, c R T$

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy.integrate as integrate
from scipy.signal import argrelextrema

In [ ]:
R = 8.314462618 #J/K/mol
N_A = 6.022e23
def gaslag(c,T):
    p=1000.*c*R*T
    return p

In [ ]:
c = np.linspace(0.001,100,1000)
T = 298 # K
plt.plot(c,gaslag(c,T))
plt.xlabel("$c$")
plt.ylabel("$p$")
plt.show()

# van der Waals ekvation för imperfekta gaser (alltså riktiga gaser)
[Van der Waals equation (Wikipedia)](https://en.wikipedia.org/wiki/Van_der_Waals_equation)

Trycket ges nu istället av:

$p = \frac{1000 c R T}{1 - 1000 c b} - a (1000 c)^2$

där $b$ är en utesluten volym och $a$ en växelverkansparameter. Om du sätter $a=0$ och $b=0$ får vi tillbaka den ideala gaslagen ovan. $b$ har enheten volym och $a$ enheten volym i kvadrat.

$b$ är ungefär lika med $\frac{4 \pi d^3}{3} N_A$, där $d$ är diametern på atomen/molekylen och $N_A$ är Avogadros tal. Vi börjar med att definiera funktionen för vdW som tar argumenten $c$ och $T$, antaget att $a$ och $b$ är kända.

In [ ]:
def vdW(c,T):
    p=1000*c*R*T/(1.-1000*c*b)-a*c*c*1e6
    return p

### Rita lite grafer för olika $a$-parametrar
Vi börjar med att definiera ett koncentrationsintervall där vi vill titta på trycket, från lågt till högt.
Notera att `vdW`-uttrycket divergerar då $1000\,c\,b$ närmar sig 1 (dvs. då nämnaren går mot noll). Det motsvarar en koncentration då atomerna börjar överlappa varandra. Vi begränsar därför $c$. $b$ beräknas för en molekyl som är 3 Å i diameter.

In [ ]:
b = 4.*np.pi/3.*np.power(3e-10,3)*N_A
cmax=1./b/1000.
c = np.logspace(np.log10(0.01),np.log10(cmax),1000,endpoint=False)
plt.plot(c,gaslag(c,T))
a = 0.
plt.plot(c,vdW(c,T))
a = 0.5
plt.plot(c,vdW(c,T))
a = 0.6
plt.plot(c,vdW(c,T))
a = 0.67
plt.plot(c,vdW(c,T))
plt.xscale('log')
plt.yscale('log')
plt.xlabel("$c$")
plt.ylabel("$p$")
plt.show()

# Notera att vi får en icke-monoton kurva
Låt oss rita graferna i $p$–$V$-planet istället.

In [ ]:
b = 4.*np.pi/3.*np.power(3e-10,3)*N_A
cmax=1./b/1000.
c = np.logspace(np.log10(0.01),np.log10(cmax),1000,endpoint=False)
plt.plot(1./c,gaslag(c,T))
a = 0.
plt.plot(1./c,vdW(c,T))
a = 0.5
plt.plot(1./c,vdW(c,T))
a = 0.6
plt.plot(1./c,vdW(c,T))
a = 0.67
plt.plot(1./c,vdW(c,T))
plt.xscale('log')
plt.yscale('log')
plt.xlabel("$c^{-1} \sim V$")
plt.ylabel("$p$")
plt.show()

In [ ]:
plt.plot(1./c,gaslag(c,T))
a = 0.
plt.plot(1./c,vdW(c,T))
a = 0.5
plt.plot(1./c,vdW(c,T))
a = 0.6
plt.plot(1./c,vdW(c,T))
a = 0.67
plt.plot(1./c,vdW(c,T))
plt.xscale('log')
plt.yscale('log')
#plt.axhline(2e6)
plt.xlabel("$c^{-1} \sim V$")
plt.ylabel("$p$")
plt.show()

# Även här ser vi en icke-monoton kurva
[Kompressibilitet (Wikipedia)](https://sv.wikipedia.org/wiki/Kompressibilitet)

Vi ser att kompressibiliteten, $\kappa = -\frac{1}{V}\frac{\partial V}{\partial p}$, är negativ för vissa volymer. Detta är ofysikaliskt, eftersom det skulle betyda att volymen ökade när vi trycker på gasen. Vad det i praktiken innebär är att lösningen fasseparerar i en utspädd fas och en koncentrerad fas.
Som ni troligtvis förstår inträffar detta så fort vi får en negativ kompressibilitet. Men till vilka koncentrationer fasseparerar vi? Och vid vilket tryck inträffar detta (kom ihåg att en fasövergång sker vid konstant tryck och temperatur)? För att ta reda på det letar vi upp lokala maxima och minima i trycket. Jämviktstrycket (för fasövergången) kommer att ligga mellan dessa två extrempunkter.

För detta ändamål använder vi `SciPy`.

# Maxwell-konstruktion
[Maxwell construction (Wikipedia)](https://en.wikipedia.org/wiki/Maxwell_construction)

För att veta vid vilket tryck fasövergången sker kan vi använda Maxwells konstruktion. Den säger att integralen $\int_{V_1}^{V_2} \left(p(V)-p_{\rm guess}\right)\,dV$ (under ett gissat tryck, mellan dess skärningspunkter med kurvan) ska vara lika med integralen $\int_{V_2}^{V_3} \left(p(V)-p_{\rm guess}\right)\,dV$ vid samma tryck, mellan dess skärningspunkter. Notera att vi får tre skärningspunkter i detta fall.
Vad vi nu gör är att testa ett antal tryck mellan de lokala maxima och minima och se när integralerna är lika med varandra (med motsatt tecken). Vi kan därmed bestämma jämviktstrycket och jämviktsdensiteterna.

### OBS: areorna ser inte lika stora ut, men det beror på vår log-log-graf

Vi definierar en ny funktion kallad `vdWp` som ger tryckskillnaden mellan `vdW` och vårt gissade tryck.

Vi använder `NumPy` för att hitta skärningspunkterna och `SciPy` för att integrera. Vi sparar summan av de två integralerna för de olika testtrycken och tar sedan det tryck som ger lika areor.

In [ ]:
def vdWp(ic,T,p0):
    p=1000./ic*R*T/(1.-1000.*b/ic)-a/ic/ic*1e6-p0
    return p

In [ ]:
ic = 1./c
press = vdW(c,T)
imax = argrelextrema(press, np.greater)
imin = argrelextrema(press, np.less)
ptest = np.linspace(press[imin],press[imax],100)
area = []
#print(press[imin])
#print(press[imax])
for i in ptest:
#    print(i)
    icross = np.argwhere(np.diff(np.sign(press - i))).flatten()
    if icross.shape[0]>2:
        int1 = integrate.quad(vdWp,ic[icross[2]],ic[icross[1]],args=(T,i))
        int2 = integrate.quad(vdWp,ic[icross[1]],ic[icross[0]],args=(T,i))
        area.append(int1[0]+int2[0])
#print(area)
zero_crossings = np.where(np.diff(np.sign(area)))[0]
print("densiteter (vätska, gas) [mol/L]:",c[icross[0]],c[icross[-1]])
plt.plot(ic,vdW(c,T))
plt.xscale('log')
plt.yscale('log')
#print(zero_crossings[0])
#print(ptest[zero_crossings[0]])
icross = np.argwhere(np.diff(np.sign(press - ptest[zero_crossings[0]]))).flatten()
#print(icross)
plt.plot([ic[icross[0]],ic[icross[-1]]],[ptest[zero_crossings[0]],ptest[zero_crossings[0]]])
icf = ic[icross[0]:icross[-1]]
plt.fill_between(icf,vdW(1./icf,T),ptest[zero_crossings[0]],alpha=0.5)
#plt.axhline(ptest[zero_crossings[0]],c='k')
#plt.plot(1./c[],vdW(c[imin],T),marker='o',c='r',mfc='w',ms=10)
plt.plot(1./c[icross[0]],vdW(c[icross[0]],T),marker='o',c='k',mfc='w',ms=10)
plt.plot(1./c[icross[-1]],vdW(c[icross[-1]],T),marker='o',c='k',mfc='w',ms=10)
plt.show()

# Uppgift

* Skapa ett fasdiagram. Slinga över olika $a$-värden för att se när vi får fasövergångar. Spara sedan jämviktstrycket och de två koncentrationerna. Rita de två koncentrationerna (den utspädda som en kurva och den koncentrerade som en annan) som funktion av $a$.

* Gör samma sak men behåll $a$ konstant och ändra istället $T$. Gör samma typ av graf (fasdiagram) men mot $T$ istället.

* Antag att $a$ har ett svagt temperaturberoende. Kommer detta att påverka ert fasdiagram? Hur? Illustrera med en graf.